# Feature Engineering

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as pyplot

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, PowerTransformer, FunctionTransformer

from sklearn.pipeline import Pipeline,make_pipeline

import pickle

import warnings
warnings.filterwarnings('ignore')

In [7]:
df = pd.read_csv('final_df.csv')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Shipment Mode              9964 non-null   object 
 1   Pack Price                 10324 non-null  float64
 2   Line Item Quantity         10324 non-null  int64  
 3   Weight (Kilograms)         10324 non-null  float64
 4   Freight Cost (USD)         10324 non-null  float64
 5   Line Item Insurance (USD)  10037 non-null  float64
dtypes: float64(4), int64(1), object(1)
memory usage: 484.1+ KB


## Clean Weight and Frieght Cost Columns

In [9]:
def cost_cols(df):

    df['Weight (Kilograms)'] = df['Weight (Kilograms)'].astype(str).str.extract(r'(\d+\.?\d*)')
    df['Weight (Kilograms)'] = pd.to_numeric(df['Weight (Kilograms)'])

    df['Freight Cost (USD)'] = df['Freight Cost (USD)'].astype(str).str.extract(r'(\d+\.?\d*)')
    df['Freight Cost (USD)'] = pd.to_numeric(df['Freight Cost (USD)'])

    df['Weight (Kilograms)'] = df['Weight (Kilograms)'].fillna(df['Weight (Kilograms)'].mean())
    df['Freight Cost (USD)'] = df['Freight Cost (USD)'].fillna(df['Freight Cost (USD)'].mean())

    return df

In [10]:
cost_cols_trf = FunctionTransformer(cost_cols)

## Missing Value Handle

In [14]:
def missing(df):

    df['Shipment Mode'] = df['Shipment Mode'].fillna('Unknown')

    return df

In [15]:
missing_trf = FunctionTransformer(missing)

## Scale and Transform using Log transform and Standard Scaler

In [33]:
num_trf = Pipeline(steps=[
    ('tranform',PowerTransformer('yeo-johnson')),
     ('scaling',StandardScaler())
])

## Categorical Encoding using OHE

In [34]:
cat_trf = Pipeline(steps=[
    ('ohe',OneHotEncoder(drop='first',sparse=False,handle_unknown='ignore'))
])

## Pipeline 

In [35]:
trf = ColumnTransformer([
    ('num_trf', num_trf, ['Pack Price', 'Line Item Quantity', 'Weight (Kilograms)', 'Freight Cost (USD)']),
    ('cat_trf', cat_trf, ['Shipment Mode'])
], remainder='passthrough')

In [36]:
fe_pipeline = make_pipeline(cost_cols_trf,missing_trf, trf)

In [37]:
fe_pipeline

Pipeline(steps=[('functiontransformer-1',
                 FunctionTransformer(func=<function cost_cols at 0x0000018C9C240680>)),
                ('functiontransformer-2',
                 FunctionTransformer(func=<function missing at 0x0000018CA6BB6A20>)),
                ('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num_trf',
                                                  Pipeline(steps=[('tranform',
                                                                   PowerTransformer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Pack Price',
                                                   'Line Item Quantity',
                                                   'Weight (Kilograms)',
                                                   'Freight Cost (USD)']),
                                                 ('cat_trf',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  ['Shipment Mode'])]))])

In [38]:
with open('fe_pipeline.pkl','wb') as f:
    pickle.dump(fe_pipeline,f)